# 🏛️ Epidemiological Archaeology: Reconstructing SARS-CoV-2 True Incidence
**Author:** Santi García-Cremades  
**Context:** Global analysis of COVID-19 unobservability bias using Topological Regularization ($H^3$).

## 📝 Introduction
During the COVID-19 pandemic, official institutional metrics—specifically the 14-day Cumulative Incidence (CI14) and daily case counts—suffered from severe unobservability bias due to testing shortages and structural healthcare limits. 

This notebook implements the **"Epidemiological Archaeology"** framework. By leveraging all-cause excess mortality (which bypasses clinical triage biases), applying a kinematic time-shift ($\tau = 20$ days), and utilizing a dynamic Infection Fatality Ratio (IFR), we retrospectively reconstruct the true infection and mortality curves for countries worldwide.

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import scipy.sparse as sparse
from scipy.sparse.linalg import spsolve
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os
import shutil

warnings.filterwarnings('ignore')

# =============================================================================
# 0. HYPERPARAMETERS
# =============================================================================
TAU_SHIFT = 20        # Clinical delay in days
LAMBDA_H3 = 5000      # Topological regularization parameter H^3

# Mapping dictionary: WMD Name -> JHU Name
COUNTRY_MAPPING = {
    "United States": "US",
    "United Kingdom": "United Kingdom",
    "South Korea": "Korea, South",
    "Taiwan": "Taiwan*",
    "Russia": "Russia",
    "Spain": "Spain"
}

# Dynamic IFR Function
def dynamic_ifr(dates):
    ifr_values = []
    for d in dates:
        if d < pd.to_datetime('2021-01-01'):
            ifr_values.append(0.0105) # Wild type
        elif d < pd.to_datetime('2021-12-01'):
            ifr_values.append(0.0060) # Alpha/Delta + Vaccines
        else:
            ifr_values.append(0.0010) # Omicron
    return np.array(ifr_values)

# H3 Regularization
def tikhonov_h3_sparse(y, lam=5000):
    n = len(y)
    if n < 4: return y
    diags = [np.ones(n), -3*np.ones(n-1), 3*np.ones(n-2), -1*np.ones(n-3)]
    D3 = sparse.diags(diags, [0, 1, 2, 3], shape=(n-3, n))
    I = sparse.eye(n)
    A = I + lam * D3.T.dot(D3)
    return spsolve(A, y)



## 🧮 1. The Mathematical Engine
The core of this model rests on filtering the noise from raw excess mortality data. We project the discrete weekly mortality signals into an $H^3$ Sobolev space using Tikhonov regularization. This forces the resulting curve to be strictly non-negative and topologically smooth, extracting the biological signal of the pandemic from the statistical noise of general mortality.

We use data from:
* **World Mortality Dataset (WMD):** For baseline and all-cause mortality.
* **Johns Hopkins University (JHU CSSE):** For official COVID-19 cases, deaths, and population demographics.

In [3]:
# =============================================================================
# 1. DOWNLOAD DATA (INCLUDING POPULATION)
# =============================================================================
print("Downloading Global Datasets (Cases, Deaths, and Populations)...")
url_wmd = "https://raw.githubusercontent.com/akarlinsky/world_mortality/main/world_mortality.csv"
url_jhu_cases = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
url_jhu_deaths = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv"
url_jhu_pop = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"

df_wmd_full = pd.read_csv(url_wmd)
df_jhu_cases_full = pd.read_csv(url_jhu_cases)
df_jhu_deaths_full = pd.read_csv(url_jhu_deaths)
df_pop_full = pd.read_csv(url_jhu_pop)


## 🌍 2. Global Processing Loop
This cell iterates over every available country in the dataset. For each nation, it:
1. Calculates the $H^3$ excess mortality.
2. Applies the clinical delay ($\tau$) and dynamic IFR to infer **True Infections**.
3. Calculates the real vs. official **CI14 (Cumulative Incidence 14-days)**.
4. Compares official COVID-19 deaths with the topological excess deaths.
5. Generates and saves a **3-Panel Figure** for visual comparison.

In [ ]:

# =============================================================================
# 2. GLOBAL PROCESSING LOOP
# =============================================================================
if not os.path.exists("Global_Plots_3Panels"):
    os.makedirs("Global_Plots_3Panels")

global_results = []
wmd_countries = df_wmd_full['country_name'].unique()

print(f"Found {len(wmd_countries)} countries in WMD. Starting global 3-panel processing...\n")

for country in wmd_countries:
    try:
        jhu_name = COUNTRY_MAPPING.get(country, country)
        
        # Check if country exists in JHU Cases
        if jhu_name not in df_jhu_cases_full['Country/Region'].values:
            continue
            
        # Get Population for CI14 calculation
        pop_data = df_pop_full[(df_pop_full['Country_Region'] == jhu_name) & (df_pop_full['Province_State'].isna())]
        if pop_data.empty or pd.isna(pop_data['Population'].values[0]):
            continue # Skip if we don't have population data
        country_pop = pop_data['Population'].values[0]
            
        # --- Process WMD (Excess Mortality) ---
        df_c_wmd = df_wmd_full[(df_wmd_full['country_name'] == country) & (df_wmd_full['year'] >= 2020)].copy()
        if df_c_wmd.empty: continue
            
        df_c_wmd['absolute_week'] = (df_c_wmd['year'] - 2020) * 52 + df_c_wmd['time']
        baseline = df_wmd_full[(df_wmd_full['country_name'] == country) & (df_wmd_full['year'] < 2020)]['deaths'].mean()
        df_c_wmd['excess_deaths'] = df_c_wmd['deaths'] - baseline
        df_c_wmd.loc[df_c_wmd['excess_deaths'] < 0, 'excess_deaths'] = 0
        
        df_c_wmd = df_c_wmd.groupby('absolute_week', as_index=False)['excess_deaths'].mean()
        df_c_wmd = df_c_wmd.sort_values('absolute_week')
        
        weeks = df_c_wmd['absolute_week'].values
        excess = df_c_wmd['excess_deaths'].values
        if len(weeks) < 4: continue
        cs = CubicSpline(weeks, excess)
        
        daily_days = np.linspace(weeks.min(), weeks.max(), int((weeks.max() - weeks.min()) * 7))
        daily_excess = cs(daily_days) / 7.0
        daily_excess[daily_excess < 0] = 0
        
        # H3 Regularization
        deaths_h3 = tikhonov_h3_sparse(daily_excess, lam=LAMBDA_H3)
        start_date = pd.to_datetime("2020-01-01") + pd.to_timedelta((weeks.min() - 1) * 7, unit='D')
        dates_h3 = [start_date + pd.to_timedelta(i, unit='D') for i in range(len(deaths_h3))]
        
        # --- Process JHU (Official Cases & Deaths) ---
        # Cases
        df_off_c = df_jhu_cases_full[df_jhu_cases_full['Country/Region'] == jhu_name].drop(columns=['Province/State', 'Country/Region', 'Lat', 'Long']).sum()
        df_off_c.index = pd.to_datetime(df_off_c.index)
        daily_cases_off = np.diff(df_off_c.values, prepend=0)
        daily_cases_off[daily_cases_off < 0] = 0
        
        # Deaths
        df_off_d = df_jhu_deaths_full[df_jhu_deaths_full['Country/Region'] == jhu_name].drop(columns=['Province/State', 'Country/Region', 'Lat', 'Long']).sum()
        df_off_d.index = pd.to_datetime(df_off_d.index)
        daily_deaths_off = np.diff(df_off_d.values, prepend=0)
        daily_deaths_off[daily_deaths_off < 0] = 0
        
        df_official = pd.DataFrame({'Date': df_off_c.index, 'Official_Cases': daily_cases_off, 'Official_Deaths': daily_deaths_off})
        
        # --- Merging & Kinematic Shift ---
        df_model = pd.DataFrame({'Date': dates_h3, 'Deaths_H3': deaths_h3})
        df_model['Infection_Date'] = df_model['Date'] - pd.to_timedelta(TAU_SHIFT, unit='D')
        
        df_final = pd.merge(df_official, df_model, left_on='Date', right_on='Infection_Date', how='inner')
        df_final['IFR'] = dynamic_ifr(df_final['Date_x'])
        df_final['True_Infections'] = df_final['Deaths_H3'] / df_final['IFR']
        
        # CI14 (IA14) Calculation using dynamic population
        df_final['CI14_Official'] = df_final['Official_Cases'].rolling(window=14, min_periods=1).sum() / (country_pop / 100000)
        df_final['CI14_Real'] = df_final['True_Infections'].rolling(window=14, min_periods=1).sum() / (country_pop / 100000)
        
        # Metrics for CSV
        total_official = df_final['Official_Cases'].sum()
        total_real = df_final['True_Infections'].sum()
        ratio = total_real / total_official if total_official > 0 else 0

        # --- NUEVAS MÉTRICAS DE MORTALIDAD ---
        total_official_deaths = df_final['Official_Deaths'].sum()
        total_h3_deaths = df_final['Deaths_H3'].sum()
        death_ratio = total_h3_deaths / total_official_deaths if total_official_deaths > 0 else 0
        
        global_results.append({
            'Country': country,
            'Population': int(country_pop),
            'Total_Official_Cases': round(total_official),
            'Total_True_Infections': round(total_real),
            'Unobservability_Ratio': round(ratio, 2),
            'Total_Official_Deaths': round(total_official_deaths),
            'Total_H3_Excess_Deaths': round(total_h3_deaths),
            'Mortality_Unobservability_Ratio': round(death_ratio, 2)
        })
        
        
        # --- 3-PANEL PLOTTING ---
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14), sharex=True)
        fig.suptitle(f"Epidemiological Archaeology: {country}", fontsize=18, fontweight='bold', y=0.98)
        
        # Panel 1: Infections
        ax1.plot(df_final['Date_x'], df_final['Official_Cases'], label='Official Cases', color='#1f77b4', linewidth=2)
        ax1.plot(df_final['Date_x'], df_final['True_Infections'], label='True Infections (H³)', color='#d62728', linestyle='--', linewidth=2)
        ax1.fill_between(df_final['Date_x'], df_final['Official_Cases'], df_final['True_Infections'], 
                         where=(df_final['True_Infections'] > df_final['Official_Cases']), color='red', alpha=0.15)
        ax1.set_ylabel("Daily Cases", fontsize=12)
        ax1.set_title("1. Daily Incidence: Official vs. Retrospective Inference", fontsize=14)
        ax1.legend(loc='upper left')
        ax1.grid(alpha=0.3)
        
        # Panel 2: CI14 (IA14)
        ax2.plot(df_final['Date_x'], df_final['CI14_Official'], label='Official CI14', color='#ff7f0e', linewidth=2)
        ax2.plot(df_final['Date_x'], df_final['CI14_Real'], label='True CI14', color='#8c564b', linestyle='-.', linewidth=2)
        ax2.axhline(250, color='black', linestyle=':', label='Extreme Risk Threshold (250)')
        ax2.fill_between(df_final['Date_x'], df_final['CI14_Official'], df_final['CI14_Real'], 
                         where=(df_final['CI14_Real'] > df_final['CI14_Official']), color='orange', alpha=0.15)
        ax2.set_ylabel("CI14 (per 100k)", fontsize=12)
        ax2.set_title("2. 14-Day Cumulative Incidence (Clinical Risk)", fontsize=14)
        ax2.legend(loc='upper left')
        ax2.grid(alpha=0.3)
        
        # Panel 3: Deaths
        # Para emparejar las fechas de las muertes, cruzamos Date_x (fecha de caso) con Date_y (fecha de muerte real H3)
        df_final = pd.merge(df_official, df_model, left_on='Date', right_on='Date', how='inner')
        
        ax3.plot(df_final['Date'], df_final['Official_Deaths'], label='Official Deaths', color='#7f7f7f', linewidth=2)
        ax3.plot(df_final['Date'], df_final['Deaths_H3'], label='Excess Deaths (H³)', color='purple', linestyle='--', linewidth=2)
        ax3.fill_between(df_final['Date'], df_final['Official_Deaths'], df_final['Deaths_H3'], 
                         where=(df_final['Deaths_H3'] > df_final['Official_Deaths']), color='purple', alpha=0.15)
        ax3.set_ylabel("Daily Deaths", fontsize=12)
        ax3.set_title("3. Mortality: Official Records vs. Topological Excess", fontsize=14)
        ax3.legend(loc='upper left')
        ax3.grid(alpha=0.3)
        
        ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
        ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        plt.xticks(rotation=45)
        plt.xlabel("Date", fontsize=12)
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.96])
        plt.savefig(f"Global_Plots_3Panels/Archaeology_3Panel_{country.replace(' ', '_')}.png", dpi=300)
        plt.close()
        
        print(f"Processed 3 Panels: {country} (Pop: {int(country_pop):,})")
        
    except Exception as e:
        print(f"Skipped {country} due to error: {e}")

# =============================================================================
# 3. EXPORT & ZIP
# =============================================================================
df_global = pd.DataFrame(global_results).sort_values(by='Unobservability_Ratio', ascending=False)
df_global.to_csv("Global_Archaeology_Results.csv", index=False)

print("\nCompressing 3-panel plots into a ZIP file...")
shutil.make_archive('Global_Plots_3Panels_Archive', 'zip', 'Global_Plots_3Panels')

print("🎉 GLOBAL BATCH COMPLETED! Check 'Global_Plots_3Panels_Archive.zip'.")